In [1]:
import pandas as pd
raw_orders = pd.DataFrame({
    "order_id": [1, 2, 2, 3, 4, 5, 6],
    "customer_id": ["C01", "C02", "C02", "C03", "C04", "C99", "C01"],
    "order_date": [
        "2026-01-01",
        "2026-01-05",
        "2026-01-05",
        "invalid",
        "2026-02-10",
        "2026-02-12",
        "2026-02-20",
    ],
    "status": ["complete", "complete", "complete", "cancelled",
               "complete", "complete", "complete"],
    "quantity": [2, 1, 1, 3, -1, 5, 2],
    "unit_price": [20, 50, 50, 15, 100, 10, 25],
})

customers = pd.DataFrame({
    "customer_id": ["C01", "C02", "C03", "C04"],
    "region": ["North", "South", "North", "West"],
    "segment": ["new", "returning", "new", "business"],
})

In [5]:
audit=pd.DataFrame({
    "check":[
        "row_count",
        "duplicate_order_ids",
        "invalid_dates",
        "negative_quantities",
        "unknown_customer_ids",
        "cancelled_orders",
        "missing_values"
    ],
    "count":[
        len(raw_orders),
        raw_orders["order_id"].duplicated().sum(),
        pd.to_datetime(
            raw_orders["order_date"],
            errors="coerce",
        ).isna().sum(),
        (raw_orders["quantity"]<0).sum(),
        (~raw_orders["customer_id"].isin(customers["customer_id"])).sum(),
        (raw_orders["status"]=="cancelled").sum(),
        raw_orders.isna().sum().sum()
    ]
})
audit

,check,count
0,row_count,7
1,duplicate_order_ids,1
2,invalid_dates,1
3,negative_quantities,1
4,unknown_customer_ids,1
5,cancelled_orders,1
6,missing_values,0


In [9]:
duplicate_orders=raw_orders[
    raw_orders["order_id"].duplicated(keep=False)
]
invalid_date=raw_orders[
    pd.to_datetime(
        raw_orders["order_date"],
        errors="coerce"
    ).isna()
]
negative_quantities=raw_orders[
    raw_orders["quantity"]<0
]
unknown_customers=raw_orders[
    ~raw_orders["customer_id"].isin(customers["customer_id"])
]
cancelled_orders=raw_orders[
    raw_orders["status"]=="cancelled"
]
print("Duplicate Orders:")
print(duplicate_orders)
print("-"*70)
print("Invalid Date:")
print(invalid_date)
print("-"*70)
print("Negative Quantities:")
print(negative_quantities)
print("-"*70)
print("Unknown Customers:")
print(unknown_customers)
print("-"*70)
print("Cancelled Orders:")
print(cancelled_orders)
print("-"*70)

Duplicate Orders:
   order_id customer_id  order_date    status  quantity  unit_price
1         2         C02  2026-01-05  complete         1          50
2         2         C02  2026-01-05  complete         1          50
----------------------------------------------------------------------
Invalid Date:
   order_id customer_id order_date     status  quantity  unit_price
3         3         C03    invalid  cancelled         3          15
----------------------------------------------------------------------
Negative Quantities:
   order_id customer_id  order_date    status  quantity  unit_price
4         4         C04  2026-02-10  complete        -1         100
----------------------------------------------------------------------
Unknown Customers:
   order_id customer_id  order_date    status  quantity  unit_price
5         5         C99  2026-02-12  complete         5          10
----------------------------------------------------------------------
Cancelled Orders:
   order_id cu

In [11]:
# Analysis Ready Table
analysis_ready_table=raw_orders.copy()

# safe date parsing
analysis_ready_table["order_date"]=pd.to_datetime(
    analysis_ready_table["order_date"],
    errors="coerce"
)

# remove cancelled orders
analysis_ready_table=analysis_ready_table[analysis_ready_table["status"]!="cancelled"]

# remove duplicate orders
analysis_ready_table=analysis_ready_table.drop_duplicates(
    subset=["order_id"],
    keep="first"
)

# flags for investigation
analysis_ready_table["negatice_quantity_flag"]=(
    analysis_ready_table["quantity"]<0
)

analysis_ready_table["unknown_customer_flags"]=(
    ~analysis_ready_table["customer_id"].isin(customers["customer_id"])
)

# revenue column
analysis_ready_table["revenue"]=(
    analysis_ready_table["quantity"]*analysis_ready_table["unit_price"]
)

analysis_ready_table=analysis_ready_table.reset_index(drop=True)
print("ANALYSIS READY TABLE")
print(analysis_ready_table)


ANALYSIS READY TABLE
   order_id customer_id order_date    status  quantity  unit_price  \
0         1         C01 2026-01-01  complete         2          20   
1         2         C02 2026-01-05  complete         1          50   
2         4         C04 2026-02-10  complete        -1         100   
3         5         C99 2026-02-12  complete         5          10   
4         6         C01 2026-02-20  complete         2          25   

   negatice_quantity_flag  unknown_customer_flags  revenue  
0                   False                   False       40  
1                   False                   False       50  
2                    True                   False     -100  
3                   False                    True       50  
4                   False                   False       50  
